## Exercise 0 solution

Use the `states` data set. Fit a model predicting energy consumed per capita (`energy`) from the percentage of residents living in metropolitan areas (`metro`). Be sure to:

1.  Examine/plot the data before fitting the model.

In [ ]:
  states_en_met <- states |> 
      select(energy, metro)

  summary(states_en_met)
  cor(states_en_met, use = "pairwise")
  qplot(x = metro, y = energy, geom = "point", data = states_en_met)

2.  Print and interpret the model `summary()`.

In [ ]:
  mod_en_met <- lm(energy ~ metro, data = states_en_met)
  summary(mod_en_met)

3.  Plot the model using `plot()` to look for deviations from modeling assumptions.

In [ ]:
  par(mfrow = c(2, 2))

  plot(mod_en_met)

4. Select one or more additional predictors to add to your model and repeat steps 1-3. Is this model significantly better than the model with `metro` as the only predictor?

In [ ]:
  states_en_met_pop_wst <- states |>
      select(energy, metro, pop, waste)

  summary(states_en_met_pop_wst)
  cor(states_en_met_pop_wst, use = "pairwise")

  mod_en_met_pop_waste <- lm(energy ~ 1 + metro + pop + waste, data = states_en_met_pop_wst)
  summary(mod_en_met_pop_waste)
  anova(mod_en_met, mod_en_met_pop_waste)

## Exercise 1 solution

Use the `states` data set.

1.  Add on to the regression equation that you created in Exercise 0 by generating an interaction term and testing the interaction.

In [ ]:
  mod_en_metro_by_waste <- lm(energy ~ 1 + metro * waste, data = states)

2.  Try adding `region` to the model. Are there significant differences across the four regions?

In [ ]:
  mod_en_region <- lm(energy ~ 1 + metro * waste + region, data = states)
  anova(mod_en_metro_by_waste, mod_en_region)

## Exercise 2 solution

Use the `NH11` data set that we loaded earlier. Note that the data are not perfectly clean and ready to be modeled. You will need to clean up at least some of the variables before fitting the model.

1.  Use `glm()` to conduct a logistic regression to predict ever worked (`everwrk`) using age (`age_p`) and marital status (`r_maritl`). Make sure you only keep the following two levels for `everwrk` (`2 No` and `1 Yes`). Hint: use the `factor()` function. Also, make sure to drop any `r_maritl` levels that do not contain observations. Hint: see `?droplevels`.

In [ ]:
  NH11 <- NH11 |>
      mutate(everwrk = factor(everwrk, levels = c("2 No", "1 Yes")),
             r_maritl = droplevels(r_maritl)
             )

  mod_wk_age_mar <- glm(everwrk ~ 1 + age_p + r_maritl, 
                        data = NH11,
                        family = binomial(link = "logit"))

  summary(mod_wk_age_mar)

2.  Predict the probability of working for each level of marital status. Hint: use `allEffects()`.

In [ ]:
  mod_wk_age_mar |> 
      allEffects() |>
      as.data.frame()

## Exercise 3 solution

Use the dataset `bh1996`:

In [ ]:
  # install.packages("multilevel")
  data(bh1996, package="multilevel")

From the data documentation:

> Variables are Leadership Climate (`LEAD`), Well-Being (`WBEING`), and Work Hours (`HRS`). The group identifier is named `GRP`.

1.  Create a null model predicting wellbeing (`WBEING`).

In [ ]:
  mod_grp0 <- lmer(WBEING ~ 1 + (1 | GRP), data = na.omit(bh1996))
  summary(mod_grp0)

2.  Calculate the ICC for your null model

In [ ]:
  0.0358 / (0.0358 + 0.7895)

3.  Run a second multi-level model that adds two individual-level predictors, average number of hours worked (`HRS`) and leadership skills (`LEAD`) to the model and interpret your output.

In [ ]:
  mod_grp1 <- lmer(WBEING ~ 1 + HRS + LEAD + (1 | GRP), data = na.omit(bh1996))
  summary(mod_grp1)

4.  Now, add a random effect of average number of hours worked (`HRS`) to the model and interpret your output. Test the significance of this random term.

In [ ]:
  mod_grp2 <- lmer(WBEING ~ 1 + HRS + LEAD + (1 + HRS | GRP), data = na.omit(bh1996))
  anova(mod_grp1, mod_grp2)